# 📖 Lab 7: Scaling Search (Deep Dive)

**Non-functional requirement:** *Low latency search (< 500ms)*

In Lab 2, we built search using `ILIKE '%keyword%'` — which forces PostgreSQL to do a **full table scan** on every query. With 5 events it's instant. With 500,000 events, it's a disaster.

## 🏗️ Architecture — Before (Lab 2)

```
┌────────┐       ┌─────────────┐       ┌────────────────┐       ┌──────────────┐
│ Client │──────>│ API Gateway │──────>│ Search Service  │──SQL─>│  PostgreSQL  │
└────────┘       └─────────────┘       └────────────────┘       └──────────────┘
                                        ILIKE '%keyword%'
                                        = full table scan
                  GET /search?term=...   O(n) 💀
```

## 🏗️ Architecture — After (ES + Cache + CDN)

```
┌────────┐     ┌───────┐     ┌─────────────┐     ┌────────────────┐
│ Client │────>│  CDN  │────>│ API Gateway  │────>│ Search Service │
└────────┘     └───┬───┘     └─────────────┘     └───────┬────────┘
               cache HIT                                  │
               (~5ms)                                     v
                                                  ┌──────────────┐
                                                  │ Redis Cache  │ cache HIT (~1ms)
                                                  │ search:{hash}│
                                                  └──────┬───────┘
                                                         │ cache MISS
                                                         v
              ┌──────────────┐    CDC    ┌───────────────────────────┐
              │  PostgreSQL  │──────────>│     Elasticsearch         │
              │ (source of   │          │  - inverted indexes        │
              │   truth)     │          │  - fuzzy search            │
              └──────────────┘          │  - relevance scoring (BM25)│
                                        │  - node query caching      │
                                        └───────────────────────────┘
```

This deep dive progresses through 3 search approaches + caching:

| # | Approach | Mechanism | Fuzzy/Typo | Relevance |
|---|----------|-----------|------------|-----------|
| 1 | **B-tree indexes** | Standard indexes | ❌ No | ❌ No |
| 2 | **Full-text search (tsvector + GIN)** | PostgreSQL built-in FTS | ❌ Limited | ✅ `ts_rank` |
| 3 | **Elasticsearch** | Dedicated search engine | ✅ Fuzzy matching | ✅ BM25 |
| 4 | **+ Redis cache + CDN** | Multi-layer caching | — | — |

## Learning Objectives

- See why `ILIKE` causes sequential scans with `EXPLAIN ANALYZE`
- Add B-tree indexes and understand their limitations
- Implement PostgreSQL full-text search with `tsvector` and GIN indexes
- Use Elasticsearch with fuzzy search and relevance scoring
- Add Redis search caching for repeated queries
- Compare performance across all approaches

## 🛠️ Setup

```bash
cd system-designs/ticketmaster
docker-compose up -d
```

Select the **"Ticketmaster (Python)"** kernel.

In [ ]:
import psycopg2
import psycopg2.extras
import time

DB_CONFIG = {
    "host": "localhost",
    "port": 5433,
    "user": "demo",
    "password": "demo",
    "database": "ticketmaster",
}

def get_connection():
    return psycopg2.connect(**DB_CONFIG)

conn = get_connection()
cur = conn.cursor()
cur.execute("SELECT COUNT(*) FROM events")
print(f"✅ Connected! {cur.fetchone()[0]} events")
cur.close()
conn.close()

## 📊 Generating Scale Data

We need enough events to see real performance differences. Let's insert 50,000 events.

In [ ]:
conn = get_connection()
cur = conn.cursor()

cur.execute("SELECT COUNT(*) FROM events")
count = cur.fetchone()[0]

if count < 1000:
    print("⏳ Inserting 50,000 events for scale testing...")
    cur.execute("""
        INSERT INTO events (name, description, event_type, venue_id, performer_id, event_date)
        SELECT 
            (ARRAY['The Eras Tour', 'Renaissance World Tour', 'Big Steppers Live',
                    'Music of the Spheres', 'Rock Night', 'Jazz Festival',
                    'Comedy Hour', 'Sports Finals', 'Pop Concert', 'Tour Stop',
                    'Summer Fest', 'Winter Gala', 'Spring Concert', 'Acoustic Night']
            )[floor(random() * 14 + 1)::int]
            || ' - ' || (ARRAY['NYC', 'LA', 'London', 'Chicago', 'Miami', 'Seattle', 
                               'Austin', 'Denver', 'Boston', 'Nashville']
            )[floor(random() * 10 + 1)::int]
            || ' #' || i,
            'An incredible live event experience. ' ||
            (ARRAY['Taylor Swift performs her greatest hits.',
                    'Beyonce brings the Renaissance tour to life.',
                    'Kendrick Lamar delivers raw lyricism.',
                    'Coldplay lights up the stadium.',
                    'A night of unforgettable comedy.',
                    'The championship game you cannot miss.']
            )[floor(random() * 6 + 1)::int],
            (ARRAY['concert', 'sports', 'comedy', 'theater'])[floor(random() * 4 + 1)::int],
            (floor(random() * 3) + 1)::int,
            (floor(random() * 8) + 1)::int,
            NOW() + (random() * interval '365 days')
        FROM generate_series(1, 50000) AS i
    """)
    conn.commit()
    print("✅ Done!")
else:
    print(f"✅ Already have {count} events.")

cur.execute("SELECT COUNT(*) FROM events")
print(f"📊 Total events: {cur.fetchone()[0]}")
cur.close()
conn.close()

## ❌ Baseline: ILIKE Full Table Scan

This is what Lab 2 does. Let's see exactly how bad it is with `EXPLAIN ANALYZE`.

In [ ]:
def search_ilike(term: str) -> dict:
    """Lab 2 approach: ILIKE with wildcards."""
    conn = get_connection()
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
    start = time.time()
    cur.execute("""
        SELECT e.id, e.name, e.event_type, e.event_date, p.name AS performer
        FROM events e
        JOIN performers p ON e.performer_id = p.id
        WHERE e.name ILIKE %s OR e.description ILIKE %s OR p.name ILIKE %s
        ORDER BY e.event_date ASC
        LIMIT 10
    """, (f"%{term}%", f"%{term}%", f"%{term}%"))
    results = cur.fetchall()
    duration = (time.time() - start) * 1000
    cur.close()
    conn.close()
    return {"count": len(results), "ms": round(duration, 2), "results": results}


# Let's see the EXPLAIN plan
conn = get_connection()
cur = conn.cursor()

print("📊 EXPLAIN ANALYZE — ILIKE search for 'Taylor'\n")
cur.execute("""
    EXPLAIN ANALYZE
    SELECT e.id, e.name
    FROM events e
    JOIN performers p ON e.performer_id = p.id
    WHERE e.name ILIKE %s OR e.description ILIKE %s OR p.name ILIKE %s
    LIMIT 10
""", ("%Taylor%", "%Taylor%", "%Taylor%"))

for row in cur.fetchall():
    print(f"  {row[0]}")

cur.close()
conn.close()

print("\n⚠️  Look for 'Seq Scan' — PostgreSQL reads EVERY row in the table.")
print("   This is O(n) — it gets slower as the table grows.")

# Measure timing
result = search_ilike("Taylor")
print(f"\n⏱️  ILIKE search for 'Taylor': {result['ms']}ms, {result['count']} results")

## 🟡 Good Approach: B-Tree Indexes

Standard B-tree indexes speed up **exact match** and **prefix** queries. Let's add them and see what they can (and can't) do.

```sql
-- These help with: WHERE name = 'Taylor Swift' or WHERE name LIKE 'Taylor%'
-- These DON'T help with: WHERE name ILIKE '%Taylor%' (substring in the middle)
```

In [ ]:
# Add B-tree indexes on searchable columns
conn = get_connection()
cur = conn.cursor()

cur.execute("CREATE INDEX IF NOT EXISTS idx_events_name_btree ON events(name)")
cur.execute("CREATE INDEX IF NOT EXISTS idx_events_type_btree ON events(event_type)")
cur.execute("CREATE INDEX IF NOT EXISTS idx_events_date_btree ON events(event_date)")
cur.execute("CREATE INDEX IF NOT EXISTS idx_performers_name_btree ON performers(name)")
conn.commit()
print("✅ B-tree indexes created")

# Test 1: Exact match — B-tree shines here
print("\n📊 EXPLAIN — exact match (B-tree CAN help)")
cur.execute("EXPLAIN ANALYZE SELECT * FROM events WHERE event_type = 'concert' LIMIT 10")
for row in cur.fetchall():
    print(f"  {row[0]}")

# Test 2: Prefix search — B-tree can help
print("\n📊 EXPLAIN — prefix search LIKE 'Eras%' (B-tree CAN help)")
cur.execute("EXPLAIN ANALYZE SELECT * FROM events WHERE name LIKE 'The Eras%' LIMIT 10")
for row in cur.fetchall():
    print(f"  {row[0]}")

# Test 3: Substring search — B-tree CANNOT help
print("\n📊 EXPLAIN — substring search ILIKE '%Eras%' (B-tree CANNOT help)")
cur.execute("EXPLAIN ANALYZE SELECT * FROM events WHERE name ILIKE '%Eras%' LIMIT 10")
for row in cur.fetchall():
    print(f"  {row[0]}")

cur.close()
conn.close()

print("\n💡 B-tree indexes work for exact match and prefix — but NOT for '%keyword%'.")
print("   Our search API lets users type any keyword → we need something better.")

## 🟢 Better Approach: PostgreSQL Full-Text Search (tsvector + GIN)

PostgreSQL has **built-in full-text search**. Instead of scanning every character with `ILIKE`, it tokenizes text into words and builds a **GIN (Generalized Inverted Index)** — similar in concept to what Elasticsearch does, but within PostgreSQL.

### How it works

```
                       tsvector                           GIN Index
"The Eras Tour - NYC"  →  'eras' 'nyc' 'tour'    →    'eras' → [row 1, row 47, row 982]
                                                        'tour' → [row 1, row 3, row 47, ...]
                                                        'nyc'  → [row 1, row 512, ...]
```

1. Text is split into **lexemes** (normalized words): "The Eras Tour" → `'eras' 'tour'`
2. A **GIN index** maps each lexeme to the rows that contain it
3. Searching for "eras" is now an **index lookup**, not a table scan

In [ ]:
# Step 1: Add a tsvector column that combines name + description
conn = get_connection()
cur = conn.cursor()

# Add the search vector column (idempotent)
cur.execute("""
    DO $$ BEGIN
        ALTER TABLE events ADD COLUMN search_vector tsvector;
    EXCEPTION WHEN duplicate_column THEN NULL;
    END $$;
""")

# Populate it: combine event name and description into one searchable vector
cur.execute("""
    UPDATE events
    SET search_vector = to_tsvector('english', coalesce(name, '') || ' ' || coalesce(description, ''))
""")

# Create a GIN index on the vector
cur.execute("CREATE INDEX IF NOT EXISTS idx_events_fts ON events USING GIN(search_vector)")

# Create a trigger so new/updated events auto-update the vector
cur.execute("""
    CREATE OR REPLACE FUNCTION update_search_vector() RETURNS trigger AS $$
    BEGIN
        NEW.search_vector := to_tsvector('english', coalesce(NEW.name, '') || ' ' || coalesce(NEW.description, ''));
        RETURN NEW;
    END;
    $$ LANGUAGE plpgsql;
""")
cur.execute("""
    DO $$ BEGIN
        CREATE TRIGGER trg_events_search_vector
        BEFORE INSERT OR UPDATE ON events
        FOR EACH ROW EXECUTE FUNCTION update_search_vector();
    EXCEPTION WHEN duplicate_object THEN NULL;
    END $$;
""")

conn.commit()

# Let's see what a tsvector looks like
cur.execute("SELECT name, search_vector FROM events WHERE id = 1")
row = cur.fetchone()
print(f"📊 Event: {row[0]}")
print(f"   tsvector: {row[1]}")

cur.close()
conn.close()

print("\n✅ Full-text search setup complete: tsvector column + GIN index + auto-update trigger")

In [ ]:
def search_fulltext(term: str) -> dict:
    """Full-text search using tsvector + GIN index."""
    conn = get_connection()
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
    start = time.time()

    # plainto_tsquery converts plain text to a tsquery
    # ts_rank scores results by relevance
    cur.execute("""
        SELECT e.id, e.name, e.event_type, e.event_date, p.name AS performer,
               ts_rank(e.search_vector, plainto_tsquery('english', %s)) AS rank
        FROM events e
        JOIN performers p ON e.performer_id = p.id
        WHERE e.search_vector @@ plainto_tsquery('english', %s)
        ORDER BY rank DESC
        LIMIT 10
    """, (term, term))
    results = cur.fetchall()
    duration = (time.time() - start) * 1000

    cur.close()
    conn.close()
    return {"count": len(results), "ms": round(duration, 2), "results": results}


# Test it
print("📊 EXPLAIN ANALYZE — full-text search for 'Eras Tour'\n")
conn = get_connection()
cur = conn.cursor()
cur.execute("""
    EXPLAIN ANALYZE
    SELECT e.id, e.name
    FROM events e
    WHERE e.search_vector @@ plainto_tsquery('english', 'Eras Tour')
    LIMIT 10
""")
for row in cur.fetchall():
    print(f"  {row[0]}")
cur.close()
conn.close()

print("\n✅ Look for 'Bitmap Index Scan on idx_events_fts' — GIN index is being used!")

# Compare results
result = search_fulltext("Eras Tour")
print(f"\n⏱️  Full-text search for 'Eras Tour': {result['ms']}ms, {result['count']} results")
for r in result["results"][:3]:
    print(f"   {r['name']} (rank: {r['rank']:.4f})")

## 📈 Performance Comparison: ILIKE vs Full-Text Search

Let's run the same searches with both approaches and compare.

In [ ]:
search_terms = ["Taylor", "Eras", "concert", "championship", "comedy"]

print(f"📊 ILIKE vs Full-Text Search — {len(search_terms)} queries\n")
print(f"{'Search Term':<20} {'ILIKE (ms)':<15} {'FTS (ms)':<15} {'Speedup'}")
print("-" * 60)

for term in search_terms:
    ilike = search_ilike(term)
    fts = search_fulltext(term)
    speedup = ilike["ms"] / fts["ms"] if fts["ms"] > 0 else float("inf")
    print(f"{term:<20} {ilike['ms']:<15} {fts['ms']:<15} {speedup:.1f}x")

print(f"\n💡 Full-text search uses a GIN index → consistent fast lookups.")
print(f"   ILIKE scans the entire table every time → gets worse with more data.")

## 🤔 Limitation: Typos and Fuzzy Search

Full-text search is great, but it doesn't handle typos. Real users make mistakes — "Tayler Swift", "Coldpaly", "Beyonse". Let's see what happens.

In [ ]:
# Typo handling comparison
typos = [
    ("Taylor",  "Tayler"),    # transposed letter
    ("Eras",    "Erras"),     # double letter
    ("Jazz",    "Jaz"),       # missing letter
    ("Coldplay","Coldpaly"),  # transposition
]

print("📊 Typo tolerance:\n")
print(f"{'Correct':<15} {'Typo':<15} {'ILIKE results':<18} {'FTS results'}")
print("-" * 60)

for correct, typo in typos:
    ilike_correct = search_ilike(correct)
    ilike_typo = search_ilike(typo)
    fts_correct = search_fulltext(correct)
    fts_typo = search_fulltext(typo)
    print(f"{correct:<15} {typo:<15} {ilike_correct['count']} → {ilike_typo['count']:<10} {fts_correct['count']} → {fts_typo['count']}")

print(f"\n⚠️  Both ILIKE and FTS return 0 results for typos.")
print(f"   ILIKE might find partial matches by accident, but FTS matches exact tokens.")
print(f"   For real typo tolerance, you need Elasticsearch's fuzzy search.")

## 🟢 Great Approach: Elasticsearch (Hands-On!)

Let's spin up a real Elasticsearch instance and see all of this in action — fuzzy search, relevance scoring, and sub-millisecond lookups.

### How it works

```
PostgreSQL ILIKE:     "Taylor" → scan all 50,000 rows  → O(n) 😰
PostgreSQL FTS:       "Taylor" → GIN index lookup       → O(log n) 🙂
                      "Tayler" → 0 results              → 😐

Elasticsearch:        "Taylor" → inverted index lookup   → O(1) 🚀
                      "Tayler" → fuzzy match → "Taylor"  → ✅ Results! 🤩
```

Elasticsearch is running in Docker on port 9200. Let's index our events and search them.

In [ ]:
from elasticsearch import Elasticsearch

es = Elasticsearch("http://localhost:9200")
print(f"✅ Elasticsearch: {es.info()['version']['number']}")

### Step 1: Create an Index with Custom Mapping

We define how Elasticsearch should analyze and store our event data. This is like defining a schema — but optimized for search.

In [ ]:
INDEX_NAME = "events"

# Delete if exists (clean start)
if es.indices.exists(index=INDEX_NAME):
    es.indices.delete(index=INDEX_NAME)

# Create index with mapping
es.indices.create(index=INDEX_NAME, body={
    "mappings": {
        "properties": {
            "name":        {"type": "text", "analyzer": "english"},
            "description": {"type": "text", "analyzer": "english"},
            "event_type":  {"type": "keyword"},
            "event_date":  {"type": "date"},
            "performer":   {"type": "text", "analyzer": "english"},
            "genre":       {"type": "keyword"},
            "venue":       {"type": "text"},
            "city":        {"type": "keyword"},
        }
    }
})

print(f"✅ Index '{INDEX_NAME}' created with custom mapping")

### Step 2: Index Events from PostgreSQL → Elasticsearch

This simulates what CDC (Change Data Capture) does: read from the source of truth (PostgreSQL) and index into Elasticsearch. In production, Debezium + Kafka would stream changes automatically.

In [ ]:
# Read all events from PostgreSQL (including the 50K generated earlier)
conn = get_connection()
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

cur.execute("""
    SELECT e.id, e.name, e.description, e.event_type, e.event_date,
           p.name AS performer, p.genre,
           v.name AS venue, v.city
    FROM events e
    JOIN performers p ON e.performer_id = p.id
    JOIN venues v ON e.venue_id = v.id
""")
events = cur.fetchall()
cur.close()
conn.close()

# Bulk index into Elasticsearch
print(f"⏳ Indexing {len(events)} events into Elasticsearch...")

from elasticsearch.helpers import bulk

actions = []
for e in events:
    actions.append({
        "_index": INDEX_NAME,
        "_id": e["id"],
        "_source": {
            "name": e["name"],
            "description": e["description"],
            "event_type": e["event_type"],
            "event_date": str(e["event_date"]),
            "performer": e["performer"],
            "genre": e["genre"],
            "venue": e["venue"],
            "city": e["city"],
        }
    })

success, errors = bulk(es, actions)
es.indices.refresh(index=INDEX_NAME)

count = es.count(index=INDEX_NAME)["count"]
print(f"✅ Indexed {success} events. ES has {count} documents.")

### Step 3: Search with Elasticsearch

Now the fun part. Let's search using ES and compare with PostgreSQL.

In [ ]:
def search_elasticsearch(term: str, fuzzy: bool = False) -> dict:
    """Search Elasticsearch with optional fuzzy matching."""
    start = time.time()

    if fuzzy:
        # Fuzzy: tolerates typos (Levenshtein distance = AUTO)
        query = {
            "multi_match": {
                "query": term,
                "fields": ["name^3", "description", "performer^2", "venue"],
                "fuzziness": "AUTO",
            }
        }
    else:
        # Standard multi-field search with relevance boosting
        # name^3 = name matches are 3x more important than description
        query = {
            "multi_match": {
                "query": term,
                "fields": ["name^3", "description", "performer^2", "venue"],
            }
        }

    result = es.search(index=INDEX_NAME, body={"query": query, "size": 10})
    duration = (time.time() - start) * 1000

    hits = []
    for hit in result["hits"]["hits"]:
        hits.append({
            "name": hit["_source"]["name"],
            "performer": hit["_source"]["performer"],
            "score": round(hit["_score"], 4),
        })

    return {"count": len(hits), "ms": round(duration, 2), "results": hits}


# Basic search
print("🔍 Elasticsearch search for 'Eras Tour':\n")
result = search_elasticsearch("Eras Tour")
print(f"   Found {result['count']} results in {result['ms']}ms\n")
for r in result["results"][:5]:
    print(f"   📊 score={r['score']:<8} {r['name']} — {r['performer']}")

### Step 4: Fuzzy Search — Typo Tolerance

This is the killer feature PostgreSQL can't match. Let's search with typos.

In [ ]:
typo_tests = [
    ("Taylor",    "Tayler"),
    ("Eras",      "Erras"),
    ("Coldplay",  "Coldpaly"),
    ("Beyoncé",   "Beyonse"),
    ("Jazz",      "Jaz"),
]

print("📊 Fuzzy Search — typo tolerance comparison:\n")
print(f"{'Correct':<12} {'Typo':<12} {'PG ILIKE':<10} {'PG FTS':<10} {'ES':<10} {'ES Fuzzy'}")
print("-" * 65)

for correct, typo in typo_tests:
    ilike_r = search_ilike(typo)
    fts_r = search_fulltext(typo)
    es_r = search_elasticsearch(typo, fuzzy=False)
    es_fuzzy_r = search_elasticsearch(typo, fuzzy=True)
    print(f"{correct:<12} {typo:<12} {ilike_r['count']:<10} {fts_r['count']:<10} {es_r['count']:<10} {es_fuzzy_r['count']}")

print(f"\n🎉 Elasticsearch with fuzziness=AUTO handles typos!")
print(f"   'Tayler' → 'Taylor', 'Coldpaly' → 'Coldplay'")
print(f"   PostgreSQL (ILIKE and FTS) returns 0 for all typos.")

### Step 5: Performance — All Three Approaches Head-to-Head

In [ ]:
search_terms = ["Taylor", "Eras", "concert", "championship", "comedy"]

print(f"📊 Performance: ILIKE vs FTS vs Elasticsearch ({len(search_terms)} queries)\n")
print(f"{'Term':<18} {'ILIKE (ms)':<14} {'FTS (ms)':<14} {'ES (ms)':<14} {'ES vs ILIKE'}")
print("-" * 72)

for term in search_terms:
    ilike = search_ilike(term)
    fts = search_fulltext(term)
    es_r = search_elasticsearch(term)
    speedup = ilike["ms"] / es_r["ms"] if es_r["ms"] > 0 else float("inf")
    print(f"{term:<18} {ilike['ms']:<14} {fts['ms']:<14} {es_r['ms']:<14} {speedup:.1f}x")

print(f"\n💡 Elasticsearch is consistently fast regardless of the query.")
print(f"   With 50K+ events, the gap only grows larger.")

### How Data Stays in Sync: CDC (Change Data Capture)

In this lab, we manually bulk-indexed events from PostgreSQL into Elasticsearch. In production, you'd use **CDC** to keep them in sync automatically:

```
Event Service ──writes──> PostgreSQL (WAL)
                               │
                          Debezium (CDC connector)
                               │
                               v
                          Kafka (message stream)
                               │
                               v
                          Elasticsearch connector
                               │
                               v
                          Elasticsearch (search index)
```

**How CDC works:**
1. PostgreSQL writes every change to its **Write-Ahead Log (WAL)** — this is how it guarantees durability
2. **Debezium** reads the WAL and converts each INSERT/UPDATE/DELETE into a Kafka event
3. An **Elasticsearch Kafka connector** consumes these events and applies them to the ES index
4. The delay is typically **milliseconds to seconds** — near-real-time

**No application code changes needed** — CDC works at the database level. The Event Service keeps writing to PostgreSQL as usual. Elasticsearch stays in sync transparently.

**Trade-off:** The search index is **eventually consistent** — there's a small window where a newly created event exists in PostgreSQL but not yet in Elasticsearch. For search, this is acceptable.

## 🚀 Deep Dive 5: Search Query Caching

Even with Elasticsearch, popular searches like "Taylor Swift" or "concerts NYC" get repeated thousands of times per second. Every request hits ES — wasteful when the results are identical.

### Two layers of caching

| Layer | What | How |
|-------|------|-----|
| **Redis search cache** | Application-level cache in front of ES | Hash query params → cache key, store results with TTL |
| **ES built-in cache** | Shard-level filter + request cache | Automatic — ES caches filter results internally |
| **CDN** | Cache non-personalized results at the edge | CloudFront/Cloudflare caches GET responses geographically |

Let's build the Redis search cache and compare.

In [ ]:
import redis
import hashlib
import json

redis_client = redis.Redis(host="localhost", port=6380, decode_responses=True)

SEARCH_CACHE_TTL = 300  # 5 minutes

def make_cache_key(**params) -> str:
    """Build a deterministic cache key from search parameters."""
    # Sort params so {a=1, b=2} and {b=2, a=1} produce the same key
    normalized = json.dumps(params, sort_keys=True)
    key_hash = hashlib.sha256(normalized.encode()).hexdigest()[:16]
    return f"search:{key_hash}"


def search_es_cached(term: str, fuzzy: bool = False) -> dict:
    """
    Search with Redis caching layer in front of Elasticsearch.
    Cache HIT → return from Redis (sub-ms).
    Cache MISS → query ES → store in Redis → return.
    """
    cache_key = make_cache_key(term=term, fuzzy=fuzzy)

    # Check cache first
    cached = redis_client.get(cache_key)
    if cached:
        result = json.loads(cached)
        result["source"] = "cache"
        return result

    # Cache miss — query Elasticsearch
    es_result = search_elasticsearch(term, fuzzy=fuzzy)

    # Store in Redis with TTL
    cache_data = {
        "count": es_result["count"],
        "ms": es_result["ms"],
        "results": es_result["results"],
    }
    redis_client.setex(cache_key, SEARCH_CACHE_TTL, json.dumps(cache_data))

    es_result["source"] = "elasticsearch"
    return es_result


# Demo: first call = ES, second call = cache
redis_client.flushdb()

print("🔍 Search 1: 'Taylor Swift' (cache MISS)\n")
r1 = search_es_cached("Taylor Swift")
print(f"   Source: {r1['source']} | Time: {r1['ms']}ms | Results: {r1['count']}")

print(f"\n🔍 Search 2: 'Taylor Swift' (cache HIT)\n")
start = time.time()
r2 = search_es_cached("Taylor Swift")
cache_ms = round((time.time() - start) * 1000, 2)
print(f"   Source: {r2['source']} | Time: {cache_ms}ms | Results: {r2['count']}")

print(f"\n🚀 Cache hit is ~{int(r1['ms'] / cache_ms) if cache_ms > 0 else '∞'}x faster than Elasticsearch!")
print(f"   Cache key: {make_cache_key(term='Taylor Swift', fuzzy=False)}")
print(f"   TTL: {redis_client.ttl(make_cache_key(term='Taylor Swift', fuzzy=False))}s")

In [ ]:
# Load test: ES-only vs ES+cache under concurrent traffic
from concurrent.futures import ThreadPoolExecutor, as_completed

def load_test_search(func, term, num_requests=100, concurrency=20):
    latencies = []
    def single(): 
        s = time.time()
        func(term)
        return (time.time() - s) * 1000
    with ThreadPoolExecutor(max_workers=concurrency) as ex:
        futures = [ex.submit(single) for _ in range(num_requests)]
        for f in as_completed(futures):
            latencies.append(f.result())
    latencies.sort()
    return {
        "avg": round(sum(latencies)/len(latencies), 2),
        "p50": round(latencies[len(latencies)//2], 2),
        "p99": round(latencies[int(len(latencies)*0.99)], 2),
    }

# Warm the cache
search_es_cached("Taylor")

print(f"📊 Load Test: 100 requests, 20 concurrent — searching 'Taylor'\n")

es_stats = load_test_search(lambda t: search_elasticsearch(t), "Taylor")
cached_stats = load_test_search(lambda t: search_es_cached(t), "Taylor")

print(f"{'Metric':<10} {'ES Direct':<15} {'ES + Cache':<15} {'Speedup'}")
print("-" * 50)
for metric in ["avg", "p50", "p99"]:
    es_v = es_stats[metric]
    c_v = cached_stats[metric]
    sp = es_v / c_v if c_v > 0 else float("inf")
    print(f"{metric.upper():<10} {es_v:<15} {c_v:<15} {sp:.1f}x")

print(f"\n💡 With caching, repeated searches skip Elasticsearch entirely.")
print(f"   The cache absorbs all the read traffic — ES only handles cold queries.")

### ES Built-in Caching + CDN

Beyond our Redis layer, Elasticsearch and CDNs add more caching:

**Elasticsearch node query cache:**
- Automatically caches filter results at the shard level
- No application code needed — just use `filter` context in queries
- Best for keyword fields: `event_type`, `city`, `genre`

**Elasticsearch request cache:**
- Caches full search responses per shard
- Enabled by default for `size=0` (aggregation-only) queries
- Can be explicitly enabled for search queries

**CDN (CloudFront / Cloudflare):**
- Cache `GET /search?...` responses at edge locations worldwide
- Same query from NYC and London gets served from different PoPs
- Only works for non-personalized results (our search is non-personalized ✅)
- Response time drops from ~50ms (ES) to ~5ms (CDN edge)

```
Request flow with all cache layers:

Client ──> CDN ──HIT?──> return (~5ms)
               ──MISS──> API GW ──> Search Service ──> Redis ──HIT?──> return (~1ms)
                                                             ──MISS──> ES ──> return (~20ms)
                                                                        ↑ ES internal query cache
```

### Cache Invalidation

The hardest part — search queries and their results aren't directly linked. Options:

| Strategy | How | Trade-off |
|----------|-----|-----------|
| **TTL-only** | Set 5-min TTL on all search caches | Simple, but stale for up to 5 min |
| **Event-based invalidation** | On new event creation, flush all search caches | Overkill — most cached searches aren't affected |
| **Tag-based invalidation** | Tag caches by event type/city, invalidate matching tags on changes | Precise, but complex to implement |
| **Short TTL + ES built-in cache** | 60s Redis TTL, rely on ES cache for warm queries | Good balance — minimal staleness, ES absorbs misses |

For a ticketing system where events are created infrequently but searched constantly, **short TTL (1-5 minutes)** is usually sufficient.

## 🧹 Cleanup

Remove the scale test data to keep the DB clean for other labs.

In [ ]:
# Clean up PostgreSQL
conn = get_connection()
cur = conn.cursor()
cur.execute("DELETE FROM events WHERE id > 5")
cur.execute("ALTER TABLE events DROP COLUMN IF EXISTS search_vector")
cur.execute("DROP TRIGGER IF EXISTS trg_events_search_vector ON events")
cur.execute("DROP FUNCTION IF EXISTS update_search_vector()")
conn.commit()
cur.execute("SELECT COUNT(*) FROM events")
print(f"✅ PostgreSQL: {cur.fetchone()[0]} events remaining.")
cur.close()
conn.close()

# Clean up Elasticsearch
if es.indices.exists(index="events"):
    es.indices.delete(index="events")
    print("✅ Elasticsearch: 'events' index deleted.")

# Clean up Redis search cache
redis_client.flushdb()
print("✅ Redis: search cache cleared.")

## 🏗️ Final Architecture — Complete System

After all 7 labs and deep dives, here's the full system:

```
┌────────┐     ┌───────┐     ┌─────────────┐
│ Client │────>│  CDN  │────>│ API Gateway  │
└────────┘     └───────┘     │ - auth       │
                              │ - rate limit │
                              │ - routing    │
                              └──────┬───────┘
                                     │
                    ┌────────────────┼────────────────────┐
                    │                │                    │
                    v                v                    v
          ┌────────────────┐ ┌──────────────┐ ┌──────────────────┐
          │ Search Service │ │Event Service │ │ Virtual Queue    │
          └───────┬────────┘ │ (N instances)│ │ (Redis ZSET)     │
                  │          └──────┬───────┘ └────────┬─────────┘
                  v                v               admit + JWT
          ┌──────────────┐ ┌──────────────┐          │
          │ Redis Cache  │ │ Redis Cache  │          v
          │ search:{hash}│ │ event:{id}   │ ┌────────────────┐
          └──────┬───────┘ └──────┬───────┘ │Booking Service │
                 │                │         └───┬──────┬─────┘
                 v                v             │      │
          ┌─────────────┐ ┌──────────────┐      │      v
          │Elasticsearch│ │  PostgreSQL  │<─────┘   Stripe
          │ fuzzy search│ │ (source of   │
          │ inverted idx│ │   truth)     │    ┌──────────────┐
          └─────────────┘ └──────────────┘    │    Redis     │
                   ^             │            │  Ticket Lock │
                   └────CDC──────┘            │  {id: user}  │
                                              │  TTL 10 min  │
                                              └──────────────┘
```

## ✅ Summary

### Search Approaches

| | ❌ ILIKE | 🟡 B-Tree Index | 🟢 Full-Text (tsvector) | 🟢 Elasticsearch |
|-|---------|-----------------|-------------------------|-------------------|
| **Mechanism** | String scan every row | Sorted tree lookup | GIN inverted index | Distributed inverted index |
| **`WHERE name ILIKE '%X%'`** | O(n) | ❌ Still O(n) | ✅ O(log n) word-based | ✅ O(1) |
| **Fuzzy / typo tolerance** | ❌ | ❌ | ❌ | ✅ Levenshtein distance |
| **Relevance ranking** | ❌ | ❌ | ✅ Basic `ts_rank` | ✅ BM25 + boosting |
| **Scale** | ~10K rows | ~100K rows | ~1M rows | Billions |

### Caching Layers (stacked)

```
Client ──> CDN (~5ms) ──> Redis Search Cache (~1ms) ──> ES (with query cache, ~20ms) ──> ES disk
```

| Layer | What Caches | TTL |
|-------|------------|-----|
| **CDN** | Full HTTP responses for popular queries | 1–5 min |
| **Redis** | Serialized search results by query hash | 1–5 min |
| **ES query cache** | Filter results per shard | Automatic (invalidated on index change) |
| **ES request cache** | Full search responses | Automatic |

### Interview guidance

- **Mid-level:** Indexes, full-text search, `EXPLAIN ANALYZE`
- **Senior:** Elasticsearch with CDC, Redis search caching
- **Staff:** CDN for non-personalized results, ES built-in caches, cache invalidation trade-offs, when NOT to add more complexity